In [15]:
from ultralytics import YOLO
import torch

# Sanity check: GPU
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Torch CUDA build:", torch.version.cuda)

CUDA available: True
GPU: NVIDIA GeForce RTX 5090
Torch CUDA build: 13.0


In [ ]:
# Install required packages for PDF processing and Gemini API
# Uncomment and run this cell ONCE to install dependencies:
# !pip install google-genai python-dotenv pdf2image easyocr

In [ ]:

# Load YOLOv12x pretrained model
model = YOLO("../pretrained/yolo12x.pt")

# Train
results = model.train(
    data="../data/cubicasa5k-2.v6i.yolov12/data.yaml",   # path to your dataset YAML
    epochs=300,
    imgsz=640,
    batch=16,            # reduce if OOM
    device=0,           # GPU 0
    amp=True,           # mixed precision (recommended)
    workers=8,
    patience=50,
    project="runs/yolo12xtrain",
    name="yolo12x_exp"
)

In [19]:
from ultralytics import YOLO
import os
import cv2
from PIL import Image

# Load model
model = YOLO("../datascience/runs/yolo12xtrain/yolo12x_exp3/weights/best.pt")

# Run prediction on `example.png` (uses GPU 0)
results = model.predict(source="../data/example.png", imgsz=640, device=0, conf=0.05, save=False)

# Prepare output directory
out_dir = "runs/predict/yolo12x_example"
os.makedirs(out_dir, exist_ok=True)

# Get annotated image (numpy BGR), convert to RGB and save
annot = results[0].plot()
annot_rgb = cv2.cvtColor(annot, cv2.COLOR_BGR2RGB)
Image.fromarray(annot_rgb).save(os.path.join(out_dir, "example.png"))

# Print brief detection summary
print(results[0].boxes)


image 1/1 /mnt/c/Users/ryan/Documents/GitHub/construction-ai/datascience/../data/example.png: 640x448 15 doors, 56 walls, 9 windows, 13.0ms
Speed: 1.1ms preprocess, 13.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 448)
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0., 0., 2., 2., 0., 1., 2., 0., 1., 0., 0., 1., 0., 1., 1., 1., 1., 0., 1., 2., 2., 1., 0., 1., 0., 2., 1., 1., 1., 1., 1., 1., 1., 2., 1., 1., 1., 2., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 0., 2.,
        1., 1.], device='cuda:0')
conf: tensor([0.6547, 0.6484, 0.6192, 0.5956, 0.5148, 0.5053, 0.4971, 0.4938, 0.4805, 0.4757, 0.4697, 0.4618, 0.4471, 0.4370, 0.4364, 0.4207, 0.4154, 0.3969, 0.3917, 0.3900, 0.3790, 0.3716, 0.3701, 0.3662, 0.3624, 0.3537, 0.3527, 0.3510, 0.3336, 0.3310, 0.3259, 0.3054, 0.3018, 0.2871, 0.2857, 0.2839, 0.2633, 0.2591, 0.2577,
        0.2573, 0.2

In [3]:
from pdf2image import convert_from_path
import easyocr
from ultralytics import YOLO
import cv2
from PIL import Image
from pathlib import Path
import numpy as np
import google.genai as genai
from google.genai import types
from dotenv import load_dotenv
import os
import re
import json
from typing import Optional, List
from pydantic import BaseModel, Field

# Load environment variables from .env file
load_dotenv()

# Configure Gemini API
# Create a .env file in your project root with: GEMINI_API_KEY=your-api-key-here
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
if GEMINI_API_KEY:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("✓ Gemini API initialized")
else:
    client = None
    print("⚠ Gemini API key not set. Skipping Gemini analysis.")
    print("  Create a .env file with: GEMINI_API_KEY=your-api-key-here")

# Initialize EasyOCR reader (English language)
print("Initializing EasyOCR reader...")
reader = easyocr.Reader(['en'], gpu=True)  # Set gpu=False if no CUDA available
print("✓ EasyOCR reader initialized")

# Load floor plan boundary detection model
boundary_model = YOLO("../datascience/runs/yolo12xtrain_floorplan_boundary/yolo12x_floorplan_boundary_exp2/weights/best.pt")
print("✓ Floor plan boundary detection model loaded")

# Load object detection model (for detecting objects within floor plans)
object_model = YOLO("../datascience/runs/yolo12xtrain/yolo12x_exp3/weights/best.pt")
print("✓ Object detection model loaded")

# ========================================
# Pydantic Models for Gemini Response
# ========================================

class ScaleInfo(BaseModel):
    """Scale information extracted from floor plan"""
    found: bool = Field(description="Whether a scale was found in the drawing")
    notation: Optional[str] = Field(default=None, description="Exact scale text as shown (e.g., '1/4\" = 1\\'-0\"')")
    format: Optional[str] = Field(default=None, description="Format type: imperial_architectural, metric_ratio, or text")
    drawing_unit: Optional[str] = Field(default=None, description="Unit on drawing (e.g., 'inch', 'mm')")
    real_unit: Optional[str] = Field(default=None, description="Real world unit (e.g., 'foot', 'meter')")
    drawing_value: Optional[float] = Field(default=None, description="Numeric value on drawing (e.g., 0.25 for 1/4 inch)")
    real_value: Optional[float] = Field(default=None, description="Numeric value in reality (e.g., 12 for 1 foot in inches)")

class FloorPlanAnalysis(BaseModel):
    """Complete analysis of a floor plan from Gemini Vision"""
    scale: ScaleInfo = Field(description="Scale information")
    title: Optional[str] = Field(default=None, description="Drawing title or name")
    room_labels: List[str] = Field(default_factory=list, description="List of room names found")
    dimensions: List[str] = Field(default_factory=list, description="List of dimension annotations (e.g., '12\\'-6\"')")
    all_text: List[str] = Field(default_factory=list, description="All other text found in the drawing")
    notes: List[str] = Field(default_factory=list, description="Notes, specifications, or legends")

# Function to analyze floor plan with Gemini using structured JSON output
def analyze_floor_plan_with_gemini(image_array, plan_type="floor plan"):
    """
    Use Gemini Vision to extract text and scale information from floor plan.
    Returns structured JSON data validated with Pydantic for reliable parsing.
    """
    if client is None:
        return None

    try:
        # Convert numpy array to PIL Image
        if isinstance(image_array, np.ndarray):
            pil_image = Image.fromarray(image_array)
        else:
            pil_image = image_array

        # Save image to bytes for upload
        import io
        img_byte_arr = io.BytesIO()
        pil_image.save(img_byte_arr, format='PNG')
        img_byte_arr.seek(0)

        # Create prompt for Gemini
        prompt = f"""Analyze this {plan_type} image and extract the following information:

1. SCALE notation - This is the most important! Look carefully for:
   - Imperial architectural format: "1/4\" = 1'-0\"", "1/8\" = 1'-0\"", "3/32\" = 1'-0\""
   - Metric ratio format: "1:100", "1:50", "1:200"
   - Text format: "Scale: 1/4 inch = 1 foot"
   - Common locations: title block, bottom right corner, drawing border, legend area

2. All visible text including:
   - Room names and labels
   - Dimension annotations
   - Drawing title or project name
   - Notes, specifications, or legends

Important scale parsing guidelines:
- For "1/4\" = 1'-0\"":
  * drawing_value = 0.25 (the fraction 1/4 as decimal)
  * real_value = 1.0 (the number of feet, NOT converted to inches)
  * drawing_unit = "inch"
  * real_unit = "foot"
  * format = "imperial_architectural"

- For "1/8\" = 1'-0\"":
  * drawing_value = 0.125 (the fraction 1/8 as decimal)
  * real_value = 1.0 (the number of feet, NOT converted to inches)
  * drawing_unit = "inch"
  * real_unit = "foot"
  * format = "imperial_architectural"

- For "1:100":
  * drawing_value = 1
  * real_value = 100
  * format = "metric_ratio"

- If scale not found, set found=false and all other scale fields to null

Examine the image carefully and extract all information."""

        # Call Gemini API with structured output using Pydantic schema
        response = client.models.generate_content(
            model='gemini-2.0-flash-exp',
            contents=[
                prompt,
                types.Part.from_bytes(
                    data=img_byte_arr.getvalue(),
                    mime_type="image/png"
                )
            ],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=FloorPlanAnalysis
            )
        )

        # Parse the response using Pydantic
        response_text = response.text.strip()

        try:
            # Parse JSON and validate with Pydantic
            data = json.loads(response_text)
            validated_data = FloorPlanAnalysis(**data)

            # Return as dict for backward compatibility
            return validated_data.model_dump()

        except json.JSONDecodeError as je:
            print(f"    ⚠ Failed to parse Gemini JSON response: {je}")
            print(f"    Response was: {response_text[:200]}...")
            # Return a fallback structure
            return FloorPlanAnalysis(
                scale=ScaleInfo(found=False, notation=None),
                title=None,
                room_labels=[],
                dimensions=[],
                all_text=[response_text] if response_text else [],
                notes=[]
            ).model_dump()

        except Exception as ve:
            print(f"    ⚠ Validation error: {ve}")
            # Return a fallback structure
            return FloorPlanAnalysis(
                scale=ScaleInfo(found=False, notation=None),
                title=None,
                room_labels=[],
                dimensions=[],
                all_text=[],
                notes=[]
            ).model_dump()

    except Exception as e:
        print(f"    ⚠ Gemini analysis failed: {e}")
        return None

def extract_scale_from_gemini_response(gemini_response):
    """Extract scale information from Gemini's structured JSON response"""
    if not gemini_response or not isinstance(gemini_response, dict):
        return None, "Not found"

    scale_info = gemini_response.get("scale", {})

    if not scale_info.get("found", False):
        return None, "Not found"

    notation = scale_info.get("notation", "")
    scale_format = scale_info.get("format", "")

    # If we have structured data, construct the scale ratio
    drawing_value = scale_info.get("drawing_value")
    real_value = scale_info.get("real_value")
    drawing_unit = scale_info.get("drawing_unit", "")
    real_unit = scale_info.get("real_unit", "")

    if drawing_value and real_value:
        # Calculate scale ratio
        if scale_format == "imperial_architectural":
            # For imperial architectural scales like "1/8\" = 1'-0\""
            # drawing_value is in inches (e.g., 0.125 for 1/8")
            # real_value is in feet (e.g., 1.0 for 1'-0")
            # We need both in the same unit (inches)

            if real_unit in ["foot", "feet"]:
                # Convert feet to inches
                real_value_inches = real_value * 12
            else:
                # Already in inches
                real_value_inches = real_value

            scale_ratio = real_value_inches / drawing_value
            return scale_ratio, notation if notation else f"{drawing_value}\" = {real_value}'"

        elif scale_format == "metric_ratio":
            # Format like 1:100 means 1 drawing unit = 100 real units
            scale_ratio = real_value / drawing_value
            return scale_ratio, notation if notation else f"1:{int(real_value)}"

        else:
            # Generic calculation
            scale_ratio = real_value / drawing_value
            return scale_ratio, notation if notation else f"{drawing_value} = {real_value}"

    # Fallback: try to parse the notation string with regex
    if notation:
        parsed_ratio, parsed_desc = parse_architectural_scale(notation)
        if parsed_ratio:
            return parsed_ratio, notation

    return None, notation if notation else "Not found"

def parse_architectural_scale(scale_string):
    """
    Parse architectural scale notation and return the scale ratio.

    Supports formats like:
    - "1/4\" = 1'-0\"" (1/4 inch = 1 foot)
    - "1:100" (ratio notation)
    - "1/8 inch = 1 foot"
    - "Scale: 1/4\" = 1'-0\""

    Returns:
        tuple: (scale_ratio, description) where scale_ratio is real_units / drawing_units
        or (None, "Not found") if parsing fails
    """
    if not scale_string or scale_string.lower() in ["not found", "none", "n/a"]:
        return None, "Not found"

    # Pattern 1: Imperial architectural scale (e.g., 1/4" = 1'-0" or 1/8" = 1'0")
    # This means 1/4 inch on paper = 1 foot in reality
    pattern1 = r'(\d+)/(\d+)\s*(?:inch|in|")\s*=\s*(\d+)\s*(?:\'|ft|feet)\s*-?\s*(\d*)\s*(?:"|in)?'
    match1 = re.search(pattern1, scale_string, re.IGNORECASE)
    if match1:
        drawing_num = float(match1.group(1))
        drawing_den = float(match1.group(2))
        real_feet = float(match1.group(3))
        real_inches = float(match1.group(4)) if match1.group(4) else 0

        # Convert to same units (inches)
        drawing_inches = drawing_num / drawing_den
        real_total_inches = real_feet * 12 + real_inches

        # Scale ratio: real size / drawing size
        scale_ratio = real_total_inches / drawing_inches
        description = f"{drawing_num}/{drawing_den}\" = {real_feet}'-{int(real_inches)}\""
        return scale_ratio, description

    # Pattern 2: Ratio notation (e.g., 1:100, 1:50)
    pattern2 = r'1\s*:\s*(\d+(?:\.\d+)?)'
    match2 = re.search(pattern2, scale_string)
    if match2:
        ratio = float(match2.group(1))
        description = f"1:{int(ratio)}"
        return ratio, description

    # Pattern 3: Simple inch to feet (e.g., "1 inch = 8 feet")
    pattern3 = r'(\d+(?:\.\d+)?)\s*(?:inch|in|")\s*=\s*(\d+(?:\.\d+)?)\s*(?:ft|feet|\')'
    match3 = re.search(pattern3, scale_string, re.IGNORECASE)
    if match3:
        drawing_inches = float(match3.group(1))
        real_feet = float(match3.group(2))
        real_inches = real_feet * 12
        scale_ratio = real_inches / drawing_inches
        description = f"{drawing_inches}\" = {real_feet}'"
        return scale_ratio, description

    return None, f"Could not parse: {scale_string}"

def calculate_real_dimensions(bbox_width_px, bbox_height_px, floor_plan_width_px, floor_plan_height_px,
                             paper_width_inches, paper_height_inches, page_width_px, page_height_px,
                             scale_ratio, dpi=300):
    """
    Calculate real-world dimensions of a bounding box.

    The calculation chain:
    1. Paper size (inches) : Page pixels : Floor plan pixels : BBox pixels
    2. Use scale to convert drawing measurements to real-world measurements

    Args:
        bbox_width_px, bbox_height_px: Bounding box dimensions in pixels
        floor_plan_width_px, floor_plan_height_px: Floor plan dimensions in pixels
        paper_width_inches, paper_height_inches: Full page paper size in inches
        page_width_px, page_height_px: Full page dimensions in pixels
        scale_ratio: Real world units / drawing units (e.g., 48 for 1/4"=1'-0")
        dpi: Dots per inch (default 300)

    Returns:
        dict: Dictionary with real-world measurements
    """
    # Step 1: Convert bbox pixels to inches on paper
    # pixels_per_inch for the full page
    ppi_width = page_width_px / paper_width_inches
    ppi_height = page_height_px / paper_height_inches

    # BBox dimensions in inches on the paper
    bbox_width_inches = bbox_width_px / ppi_width
    bbox_height_inches = bbox_height_px / ppi_height

    # Step 2: Apply scale to get real-world dimensions
    if scale_ratio:
        real_width_inches = bbox_width_inches * scale_ratio
        real_height_inches = bbox_height_inches * scale_ratio

        # Convert to feet and inches
        real_width_feet = int(real_width_inches // 12)
        real_width_remaining_inches = real_width_inches % 12

        real_height_feet = int(real_height_inches // 12)
        real_height_remaining_inches = real_height_inches % 12

        return {
            'bbox_pixels': (bbox_width_px, bbox_height_px),
            'bbox_inches_on_paper': (bbox_width_inches, bbox_height_inches),
            'real_inches': (real_width_inches, real_height_inches),
            'real_feet_inches': (
                (real_width_feet, real_width_remaining_inches),
                (real_height_feet, real_height_remaining_inches)
            ),
            'real_feet_decimal': (real_width_inches / 12, real_height_inches / 12),
            'scale_ratio': scale_ratio
        }
    else:
        return {
            'bbox_pixels': (bbox_width_px, bbox_height_px),
            'bbox_inches_on_paper': (bbox_width_inches, bbox_height_inches),
            'real_inches': None,
            'real_feet_inches': None,
            'real_feet_decimal': None,
            'scale_ratio': None,
            'note': 'No scale information available'
        }

def format_dimension(feet, inches):
    """Format dimension as architectural notation (e.g., 10'-6\")"""
    if feet == 0:
        return f"{inches:.2f}\""
    else:
        return f"{feet}'-{inches:.2f}\""

def create_numbered_annotation(image, detection_boxes, class_names, output_path, objects_by_class):
    """
    Create an annotated image with numbered labels for each detected object.

    Args:
        image: numpy array of the image
        detection_boxes: YOLO detection boxes
        class_names: dictionary mapping class IDs to names
        output_path: path to save the annotated image
        objects_by_class: dictionary grouping objects by class (for consistent numbering)

    Returns:
        Annotated image as numpy array
    """
    import cv2

    # Create a copy of the image
    annotated_img = image.copy()

    # Create a global numbering map
    object_numbering = {}
    current_number_by_class = {}

    # Initialize counters for each class
    for class_name in objects_by_class.keys():
        current_number_by_class[class_name] = 1

    # First pass: assign numbers to each object
    for obj_idx, obj_box in enumerate(detection_boxes):
        obj_cls = int(obj_box.cls[0].cpu().numpy())
        obj_class_name = class_names[obj_cls]

        # Assign number
        object_numbering[obj_idx] = {
            'class': obj_class_name,
            'number': current_number_by_class[obj_class_name]
        }
        current_number_by_class[obj_class_name] += 1

    # Second pass: draw boxes and labels
    for obj_idx, obj_box in enumerate(detection_boxes):
        # Get bounding box coordinates
        x1, y1, x2, y2 = obj_box.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

        # Get object info
        obj_conf = obj_box.conf[0].cpu().numpy()
        obj_cls = int(obj_box.cls[0].cpu().numpy())
        obj_class_name = class_names[obj_cls]
        obj_number = object_numbering[obj_idx]['number']

        # Define colors for different classes
        class_colors = {
            'wall': (255, 100, 100),      # Light red
            'window': (100, 255, 100),     # Light green
            'door': (100, 100, 255),       # Light blue
            'room': (255, 255, 100),       # Yellow
        }

        # Get color for this class (default to cyan if not in dict)
        color = class_colors.get(obj_class_name.lower(), (100, 255, 255))

        # Draw bounding box
        cv2.rectangle(annotated_img, (x1, y1), (x2, y2), color, 2)

        # Create label text
        label = f"{obj_class_name} #{obj_number}"

        # Calculate text size
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.6
        thickness = 2
        (text_width, text_height), baseline = cv2.getTextSize(label, font, font_scale, thickness)

        # Draw label background
        label_y = y1 - 10 if y1 - 10 > text_height else y1 + text_height + 10
        cv2.rectangle(annotated_img,
                     (x1, label_y - text_height - 5),
                     (x1 + text_width + 5, label_y + baseline),
                     color, -1)

        # Draw label text
        cv2.putText(annotated_img, label, (x1 + 2, label_y - 2),
                   font, font_scale, (0, 0, 0), thickness, cv2.LINE_AA)

        # Draw confidence score (smaller text)
        conf_label = f"{obj_conf:.2f}"
        font_scale_small = 0.4
        thickness_small = 1
        conf_y = y2 + 15
        cv2.putText(annotated_img, conf_label, (x1 + 2, conf_y),
                   font, font_scale_small, color, thickness_small, cv2.LINE_AA)

    return annotated_img

# Standard architectural paper sizes (width x height in inches)
PAPER_SIZES = {
    # ARCH series (landscape orientation typically)
    'ARCH A': (9, 12),
    'ARCH B': (12, 18),
    'ARCH C': (18, 24),
    'ARCH D': (24, 36),
    'ARCH E': (36, 48),
    'ARCH E1': (30, 42),

    # ANSI series
    'ANSI A (Letter)': (8.5, 11),
    'ANSI B (Tabloid)': (11, 17),
    'ANSI C': (17, 22),
    'ANSI D': (22, 34),
    'ANSI E': (34, 44),

    # ISO A series (metric)
    'A4': (8.27, 11.69),
    'A3': (11.69, 16.54),
    'A2': (16.54, 23.39),
    'A1': (23.39, 33.11),
    'A0': (33.11, 46.81),
}

def detect_paper_size(width_px, height_px, dpi=300):
    """
    Detect paper size based on pixel dimensions and DPI.

    Args:
        width_px: Width in pixels
        height_px: Height in pixels
        dpi: Dots per inch used for conversion

    Returns:
        tuple: (paper_size_name, width_inches, height_inches, orientation)
    """
    # Convert pixels to inches
    width_in = width_px / dpi
    height_in = height_px / dpi

    # Determine orientation
    if width_in > height_in:
        orientation = "Landscape"
        # For landscape, we'll check both orientations of standard sizes
        current_dims = (width_in, height_in)
    else:
        orientation = "Portrait"
        current_dims = (width_in, height_in)

    # Find closest match (allow 5% tolerance)
    tolerance = 0.05
    best_match = None
    min_error = float('inf')

    for size_name, (std_w, std_h) in PAPER_SIZES.items():
        # Check both orientations of the standard size
        for dims in [(std_w, std_h), (std_h, std_w)]:
            w_error = abs(current_dims[0] - dims[0]) / dims[0]
            h_error = abs(current_dims[1] - dims[1]) / dims[1]
            total_error = w_error + h_error

            if w_error < tolerance and h_error < tolerance:
                if total_error < min_error:
                    min_error = total_error
                    best_match = size_name

    if best_match:
        return (best_match, width_in, height_in, orientation)
    else:
        # Custom size
        return (f"Custom ({width_in:.1f}\" x {height_in:.1f}\")", width_in, height_in, orientation)

# Setup paths
pdf_folder = Path("../data")
output_folder = Path("runs/predict/pdf_analysis_complete")
output_folder.mkdir(parents=True, exist_ok=True)
for pdf_file in pdf_folder.glob("*.pdf"):
    print(f"\n{'='*60}")
    print(f"Processing: {pdf_file.name}")
    print(f"{'='*60}\n")

    # Convert PDF to images (one per page)
    print("Converting PDF to images...")
    pages = convert_from_path(pdf_file, dpi=300)
    print(f"✓ Converted {len(pages)} page(s)")

    # Process each page
    for page_num, page_img in enumerate(pages, start=1):
        print(f"\n{'='*60}")
        print(f"Page {page_num}")
        print(f"{'='*60}")

        # Save original page
        page_path = output_folder / f"{pdf_file.stem}_page{page_num}_original.png"
        page_img.save(page_path)
        print(f"✓ Saved original page: {page_path.name}")

        # Detect paper size
        img_width, img_height = page_img.size  # PIL Image dimensions
        paper_size, width_inches, height_inches, orientation = detect_paper_size(img_width, img_height, dpi=300)

        print(f"\n📐 Paper Size Analysis:")
        print(f"  Dimensions: {img_width} x {img_height} pixels")
        print(f"  Size: {width_inches:.2f}\" x {height_inches:.2f}\"")
        print(f"  Detected as: {paper_size}")
        print(f"  Orientation: {orientation}")

        # Save paper size info to file
        paper_size_path = output_folder / f"{pdf_file.stem}_page{page_num}_paper_info.txt"
        with open(paper_size_path, 'w', encoding='utf-8') as f:
            f.write(f"=== Paper Size Information ===\n\n")
            f.write(f"Page Number: {page_num}\n")
            f.write(f"Dimensions (pixels): {img_width} x {img_height}\n")
            f.write(f"Dimensions (inches): {width_inches:.2f}\" x {height_inches:.2f}\"\n")
            f.write(f"DPI: 300\n")
            f.write(f"Paper Size: {paper_size}\n")
            f.write(f"Orientation: {orientation}\n\n")

            # Add common architectural sizes for reference
            f.write("=== Common Architectural Paper Sizes ===\n\n")
            f.write("ARCH Series:\n")
            f.write("  ARCH A: 9\" x 12\"\n")
            f.write("  ARCH B: 12\" x 18\"\n")
            f.write("  ARCH C: 18\" x 24\"\n")
            f.write("  ARCH D: 24\" x 36\"\n")
            f.write("  ARCH E: 36\" x 48\"\n")
            f.write("  ARCH E1: 30\" x 42\"\n")

        print(f"✓ Paper size info saved: {paper_size_path.name}")

        # ========================================
        # Step 1: OCR on the whole page
        # ========================================
        print("\n[1/3] Running OCR on entire page...")
        page_np = np.array(page_img)
        ocr_results = reader.readtext(page_np)

        # Extract all text
        all_text_lines = [detection[1] for detection in ocr_results]
        all_text = '\n'.join(all_text_lines)

        # Save whole page OCR results
        page_ocr_path = output_folder / f"{pdf_file.stem}_page{page_num}_full_ocr.txt"
        with open(page_ocr_path, 'w', encoding='utf-8') as f:
            f.write("=== Full Page OCR Results ===\n\n")
            f.write(all_text)
            f.write("\n\n=== Detailed Results with Confidence ===\n")
            for detection in ocr_results:
                bbox, text, confidence = detection
                f.write(f"Text: {text} (confidence: {confidence:.3f})\n")

        print(f"✓ Detected {len(all_text_lines)} text regions on full page")
        print(f"✓ Full page OCR saved: {page_ocr_path.name}")

        # Run Gemini analysis on full page
        if client:
            print("\n[1b/3] Running Gemini analysis on full page...")
            gemini_full_page = analyze_floor_plan_with_gemini(page_np, "architectural drawing page")

            if gemini_full_page:
                # Save Gemini analysis (as formatted JSON)
                gemini_page_path = output_folder / f"{pdf_file.stem}_page{page_num}_gemini_analysis.txt"
                with open(gemini_page_path, 'w', encoding='utf-8') as f:
                    f.write("=== Gemini Full Page Analysis ===\n\n")
                    f.write(json.dumps(gemini_full_page, indent=2))

                # Extract and display scale
                page_scale_ratio, page_scale_notation = extract_scale_from_gemini_response(gemini_full_page)
                print(f"✓ Gemini analysis complete")
                print(f"  Page Scale: {page_scale_notation}")
                if page_scale_ratio:
                    print(f"  Scale Ratio: 1:{page_scale_ratio:.2f}")
                print(f"✓ Gemini analysis saved: {gemini_page_path.name}")

        # ========================================
        # Step 2: Detect floor plan boundaries
        # ========================================
        print("\n[2/3] Detecting floor plan boundaries...")
        boundary_results = boundary_model.predict(
            source=str(page_path),
            imgsz=640,
            device=0,
            conf=0.80,
            save=False
        )

        detections = boundary_results[0].boxes
        num_floor_plans = len(detections)
        print(f"✓ Detected {num_floor_plans} floor plan(s)")

        # Save page with floor plan boundaries annotated
        boundary_annot = boundary_results[0].plot()
        boundary_annot_rgb = cv2.cvtColor(boundary_annot, cv2.COLOR_BGR2RGB)
        boundary_annot_path = output_folder / f"{pdf_file.stem}_page{page_num}_boundaries.png"
        Image.fromarray(boundary_annot_rgb).save(boundary_annot_path)
        print(f"✓ Saved boundary detection: {boundary_annot_path.name}")

        # ========================================
        # Step 3: Extract and analyze each floor plan
        # ========================================
        if num_floor_plans == 0:
            print("⚠ No floor plans detected on this page")
            continue

        print(f"\n[3/3] Analyzing {num_floor_plans} floor plan(s)...")
        img_height, img_width = page_np.shape[:2]

        for idx, box in enumerate(detections):
            print(f"\n--- Floor Plan {idx+1}/{num_floor_plans} ---")

            # Get bounding box coordinates
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = box.conf[0].cpu().numpy()

            # Convert to integers
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            print(f"  Boundary confidence: {conf:.3f}")

            # Crop floor plan from page
            floor_plan_crop = page_np[y1:y2, x1:x2]

            # Save cropped floor plan
            crop_path = output_folder / f"{pdf_file.stem}_page{page_num}_floorplan{idx+1}.png"
            Image.fromarray(floor_plan_crop).save(crop_path)
            print(f"  ✓ Saved cropped floor plan: {crop_path.name}")

            # Run object detection on this floor plan
            print(f"  Running object detection...")
            object_results = object_model.predict(
                source=str(crop_path),
                imgsz=640,
                device=0,
                conf=0.05,
                save=False
            )

            num_objects = len(object_results[0].boxes)
            print(f"  ✓ Detected {num_objects} object(s) in floor plan")

            # Get class names from model
            class_names = object_model.names

            # Group objects by class first (for consistent numbering)
            objects_by_class = {}
            for obj_idx, obj_box in enumerate(object_results[0].boxes):
                obj_cls = int(obj_box.cls[0].cpu().numpy())
                obj_class_name = class_names[obj_cls]

                if obj_class_name not in objects_by_class:
                    objects_by_class[obj_class_name] = []

                objects_by_class[obj_class_name].append(obj_idx)

            # Create numbered annotation
            numbered_annot = create_numbered_annotation(
                floor_plan_crop,
                object_results[0].boxes,
                class_names,
                None,  # We'll save it below
                objects_by_class
            )

            # Save numbered annotated floor plan
            numbered_annot_path = output_folder / f"{pdf_file.stem}_page{page_num}_floorplan{idx+1}_objects_numbered.png"
            Image.fromarray(numbered_annot).save(numbered_annot_path)
            print(f"  ✓ Saved numbered object detection: {numbered_annot_path.name}")

            # Also save the standard YOLO annotation for comparison
            object_annot = object_results[0].plot()
            object_annot_rgb = cv2.cvtColor(object_annot, cv2.COLOR_BGR2RGB)
            object_annot_path = output_folder / f"{pdf_file.stem}_page{page_num}_floorplan{idx+1}_objects.png"
            Image.fromarray(object_annot_rgb).save(object_annot_path)
            print(f"  ✓ Saved standard object detection: {object_annot_path.name}")

            # Run OCR on this floor plan
            print(f"  Running OCR on floor plan...")
            floor_plan_ocr = reader.readtext(floor_plan_crop)
            floor_plan_text_lines = [detection[1] for detection in floor_plan_ocr]
            floor_plan_text = '\n'.join(floor_plan_text_lines)

            # Save floor plan OCR results
            if floor_plan_text.strip():
                floor_plan_ocr_path = output_folder / f"{pdf_file.stem}_page{page_num}_floorplan{idx+1}_ocr.txt"
                with open(floor_plan_ocr_path, 'w', encoding='utf-8') as f:
                    f.write(f"=== Floor Plan {idx+1} OCR Results ===\n\n")
                    f.write(floor_plan_text)
                    f.write("\n\n=== Detailed Results ===\n")
                    for detection in floor_plan_ocr:
                        bbox, text, confidence = detection
                        f.write(f"Text: {text} (confidence: {confidence:.3f})\n")

                print(f"  ✓ Floor plan OCR saved: {floor_plan_ocr_path.name} ({len(floor_plan_text_lines)} text regions)")
            else:
                print(f"  ⚠ No text detected in this floor plan")

            # Run Gemini analysis on this floor plan
            gemini_floor_plan = None
            if client:
                print(f"  Running Gemini analysis on floor plan...")
                gemini_floor_plan = analyze_floor_plan_with_gemini(floor_plan_crop, f"floor plan {idx+1}")

                if gemini_floor_plan:
                    # Save Gemini analysis (as formatted JSON)
                    gemini_floor_plan_path = output_folder / f"{pdf_file.stem}_page{page_num}_floorplan{idx+1}_gemini.txt"
                    with open(gemini_floor_plan_path, 'w', encoding='utf-8') as f:
                        f.write(f"=== Floor Plan {idx+1} Gemini Analysis ===\n\n")
                        f.write(json.dumps(gemini_floor_plan, indent=2))

                    # Extract and display scale
                    floor_plan_scale_ratio, floor_plan_scale_notation = extract_scale_from_gemini_response(gemini_floor_plan)
                    print(f"  ✓ Gemini analysis saved: {gemini_floor_plan_path.name}")
                    print(f"    Scale detected: {floor_plan_scale_notation}")
                    if floor_plan_scale_ratio:
                        print(f"    Scale ratio: 1:{floor_plan_scale_ratio:.2f}")

            # ========================================
            # Step 4: Calculate Real-World Dimensions
            # ========================================
            print(f"\n  [4/4] Calculating real-world dimensions...")

            # Try to get scale from Gemini analysis
            scale_ratio = None
            scale_description = "Not found"

            if gemini_floor_plan:
                scale_ratio, scale_description = extract_scale_from_gemini_response(gemini_floor_plan)

            # Floor plan dimensions
            floor_plan_width_px = x2 - x1
            floor_plan_height_px = y2 - y1

            # Calculate floor plan real dimensions
            floor_plan_dims = calculate_real_dimensions(
                floor_plan_width_px, floor_plan_height_px,
                floor_plan_width_px, floor_plan_height_px,
                width_inches, height_inches,
                img_width, img_height,
                scale_ratio
            )

            # Create measurements file
            measurements_path = output_folder / f"{pdf_file.stem}_page{page_num}_floorplan{idx+1}_measurements.txt"
            with open(measurements_path, 'w', encoding='utf-8') as f:
                f.write(f"=== Floor Plan {idx+1} - Real-World Measurements ===\n\n")
                f.write(f"Document Information:\n")
                f.write(f"  Paper Size: {paper_size}\n")
                f.write(f"  Paper Dimensions: {width_inches:.2f}\" x {height_inches:.2f}\"\n")
                f.write(f"  Page Resolution: {img_width} x {img_height} pixels\n")
                f.write(f"  DPI: 300\n")
                f.write(f"  Pixels per Inch: {img_width/width_inches:.2f} x {img_height/height_inches:.2f}\n\n")

                f.write(f"Scale Information:\n")
                f.write(f"  Scale Notation: {scale_description}\n")
                if scale_ratio:
                    f.write(f"  Scale Ratio: 1:{scale_ratio:.2f} (real/drawing)\n")
                    f.write(f"  Meaning: 1 inch on paper = {scale_ratio:.2f} inches in reality\n")
                    f.write(f"           1 inch on paper = {scale_ratio/12:.2f} feet in reality\n\n")
                else:
                    f.write(f"  WARNING: No valid scale found. Real-world dimensions cannot be calculated.\n\n")

                f.write(f"Floor Plan Dimensions:\n")
                f.write(f"  Pixels: {floor_plan_width_px} x {floor_plan_height_px} px\n")
                f.write(f"  On Paper: {floor_plan_dims['bbox_inches_on_paper'][0]:.3f}\" x {floor_plan_dims['bbox_inches_on_paper'][1]:.3f}\"\n")

                if floor_plan_dims['real_inches']:
                    w_ft, w_in = floor_plan_dims['real_feet_inches'][0]
                    h_ft, h_in = floor_plan_dims['real_feet_inches'][1]
                    f.write(f"  Real World: {format_dimension(w_ft, w_in)} x {format_dimension(h_ft, h_in)}\n")
                    f.write(f"  Real World (decimal): {floor_plan_dims['real_feet_decimal'][0]:.2f}' x {floor_plan_dims['real_feet_decimal'][1]:.2f}'\n")
                    f.write(f"  Real World (inches): {floor_plan_dims['real_inches'][0]:.2f}\" x {floor_plan_dims['real_inches'][1]:.2f}\"\n")
                    f.write(f"  Real World Area: {floor_plan_dims['real_feet_decimal'][0] * floor_plan_dims['real_feet_decimal'][1]:.2f} sq ft\n\n")
                else:
                    f.write(f"  Real World: Cannot calculate (no scale)\n\n")

                # Analyze detected objects
                if num_objects > 0:
                    f.write(f"\n{'='*60}\n")
                    f.write(f"Detected Objects: {num_objects}\n")
                    f.write(f"{'='*60}\n\n")
                    f.write(f"NOTE: Object numbers correspond to labels in the annotated image:\n")
                    f.write(f"      {pdf_file.stem}_page{page_num}_floorplan{idx+1}_objects_numbered.png\n\n")

                    # Write measurements for each class
                    for class_name in sorted(objects_by_class.keys()):
                        objects_in_class = objects_by_class[class_name]
                        f.write(f"\n{class_name.upper()} ({len(objects_in_class)} detected)\n")
                        f.write(f"{'-'*60}\n")

                        obj_number = 1
                        for obj_list_idx in objects_in_class:
                            obj_box = object_results[0].boxes[obj_list_idx]

                            # Get object information
                            obj_x1, obj_y1, obj_x2, obj_y2 = obj_box.xyxy[0].cpu().numpy()
                            obj_conf = obj_box.conf[0].cpu().numpy()

                            # Calculate dimensions
                            obj_width_px = obj_x2 - obj_x1
                            obj_height_px = obj_y2 - obj_y1

                            # Calculate real-world dimensions
                            obj_dims = calculate_real_dimensions(
                                obj_width_px, obj_height_px,
                                floor_plan_width_px, floor_plan_height_px,
                                width_inches, height_inches,
                                img_width, img_height,
                                scale_ratio
                            )

                            f.write(f"\n  {class_name} #{obj_number}:\n")
                            f.write(f"    Confidence: {obj_conf:.3f}\n")
                            f.write(f"    Pixels: {obj_dims['bbox_pixels'][0]:.1f} x {obj_dims['bbox_pixels'][1]:.1f} px\n")
                            f.write(f"    On Paper: {obj_dims['bbox_inches_on_paper'][0]:.3f}\" x {obj_dims['bbox_inches_on_paper'][1]:.3f}\"\n")

                            if obj_dims['real_inches']:
                                w_ft, w_in = obj_dims['real_feet_inches'][0]
                                h_ft, h_in = obj_dims['real_feet_inches'][1]
                                f.write(f"    Real Size: {format_dimension(w_ft, w_in)} x {format_dimension(h_ft, h_in)}\n")
                                f.write(f"    Real Size (inches): {obj_dims['real_inches'][0]:.2f}\" x {obj_dims['real_inches'][1]:.2f}\"\n")

                                # For walls, windows, doors - provide typical dimension
                                if class_name.lower() in ['wall', 'walls']:
                                    length_ft = max(obj_dims['real_feet_decimal'])
                                    thickness_in = min(obj_dims['real_inches'])
                                    f.write(f"    Wall Length: {length_ft:.2f}'\n")
                                    f.write(f"    Wall Thickness: {thickness_in:.2f}\"\n")

                                elif class_name.lower() in ['window', 'windows']:
                                    width_in = obj_dims['real_inches'][0]
                                    height_in = obj_dims['real_inches'][1]
                                    f.write(f"    Window Width: {width_in:.2f}\" ({width_in/12:.2f}')\n")
                                    f.write(f"    Window Height: {height_in:.2f}\" ({height_in/12:.2f}')\n")

                                elif class_name.lower() in ['door', 'doors']:
                                    width_in = obj_dims['real_inches'][0]
                                    height_in = obj_dims['real_inches'][1]
                                    f.write(f"    Door Width: {width_in:.2f}\" ({width_in/12:.2f}')\n")
                                    f.write(f"    Door Height: {height_in:.2f}\" ({height_in/12:.2f}')\n")
                            else:
                                f.write(f"    Real Size: Cannot calculate (no scale)\n")

                            obj_number += 1

                    # Summary statistics
                    f.write(f"\n\n{'='*60}\n")
                    f.write(f"SUMMARY STATISTICS\n")
                    f.write(f"{'='*60}\n\n")

                    for class_name in sorted(objects_by_class.keys()):
                        objects_in_class = objects_by_class[class_name]
                        f.write(f"\n{class_name.upper()}:\n")
                        f.write(f"  Count: {len(objects_in_class)}\n")

                        # Calculate stats from the actual objects
                        if scale_ratio:
                            widths = []
                            heights = []

                            for obj_list_idx in objects_in_class:
                                obj_box = object_results[0].boxes[obj_list_idx]
                                obj_x1, obj_y1, obj_x2, obj_y2 = obj_box.xyxy[0].cpu().numpy()
                                obj_width_px = obj_x2 - obj_x1
                                obj_height_px = obj_y2 - obj_y1

                                obj_dims = calculate_real_dimensions(
                                    obj_width_px, obj_height_px,
                                    floor_plan_width_px, floor_plan_height_px,
                                    width_inches, height_inches,
                                    img_width, img_height,
                                    scale_ratio
                                )

                                if obj_dims['real_inches']:
                                    widths.append(obj_dims['real_inches'][0])
                                    heights.append(obj_dims['real_inches'][1])

                            if widths and heights:
                                avg_width = np.mean(widths)
                                avg_height = np.mean(heights)
                                min_width = np.min(widths)
                                max_width = np.max(widths)
                                min_height = np.min(heights)
                                max_height = np.max(heights)

                                f.write(f"  Average Size: {avg_width:.2f}\" x {avg_height:.2f}\" ({avg_width/12:.2f}' x {avg_height/12:.2f}')\n")
                                f.write(f"  Width Range: {min_width:.2f}\" - {max_width:.2f}\"\n")
                                f.write(f"  Height Range: {min_height:.2f}\" - {max_height:.2f}\"\n")

                                # Special calculations for specific classes
                                if class_name.lower() in ['wall', 'walls']:
                                    # Calculate total wall length
                                    total_length_ft = 0
                                    for obj_list_idx in objects_in_class:
                                        obj_box = object_results[0].boxes[obj_list_idx]
                                        obj_x1, obj_y1, obj_x2, obj_y2 = obj_box.xyxy[0].cpu().numpy()
                                        obj_width_px = obj_x2 - obj_x1
                                        obj_height_px = obj_y2 - obj_y1

                                        obj_dims = calculate_real_dimensions(
                                            obj_width_px, obj_height_px,
                                            floor_plan_width_px, floor_plan_height_px,
                                            width_inches, height_inches,
                                            img_width, img_height,
                                            scale_ratio
                                        )

                                        if obj_dims['real_feet_decimal']:
                                            total_length_ft += max(obj_dims['real_feet_decimal'])

                                    f.write(f"  Total Wall Length: {total_length_ft:.2f}' ({total_length_ft} linear feet)\n")

                else:
                    f.write(f"\nNo objects detected in this floor plan.\n")

            print(f"  ✓ Measurements saved: {measurements_path.name}")

            # Print summary to console
            if scale_ratio:
                print(f"\n  📏 Floor Plan Real Dimensions:")
                if floor_plan_dims['real_feet_inches']:
                    w_ft, w_in = floor_plan_dims['real_feet_inches'][0]
                    h_ft, h_in = floor_plan_dims['real_feet_inches'][1]
                    print(f"     {format_dimension(w_ft, w_in)} x {format_dimension(h_ft, h_in)}")
                    print(f"     Area: {floor_plan_dims['real_feet_decimal'][0] * floor_plan_dims['real_feet_decimal'][1]:.2f} sq ft")

                if num_objects > 0:
                    print(f"\n  📐 Detected Objects Summary:")
                    for class_name in sorted(objects_by_class.keys()):
                        objects = objects_by_class[class_name]
                        print(f"     {class_name}: {len(objects)} items")
            else:
                print(f"\n  ⚠ No scale information - real-world dimensions cannot be calculated")
                print(f"    Pixel and paper measurements are available in the measurements file")

print("\n" + "="*60)
print("Processing complete!")
print(f"Results saved to: {output_folder.absolute()}")
print("="*60)


✓ Gemini API initialized
Initializing EasyOCR reader...
✓ EasyOCR reader initialized
✓ Floor plan boundary detection model loaded
✓ Object detection model loaded

Processing: example.pdf

Converting PDF to images...
✓ Converted 1 page(s)

Page 1
✓ Saved original page: example_page1_original.png

📐 Paper Size Analysis:
  Dimensions: 5100 x 3300 pixels
  Size: 17.00" x 11.00"
  Detected as: ANSI B (Tabloid)
  Orientation: Landscape
✓ Paper size info saved: example_page1_paper_info.txt

[1/3] Running OCR on entire page...
✓ Detected 178 text regions on full page
✓ Full page OCR saved: example_page1_full_ocr.txt

[1b/3] Running Gemini analysis on full page...
✓ Gemini analysis complete
  Page Scale: 1/8"=1'-0"
  Scale Ratio: 1:96.00
✓ Gemini analysis saved: example_page1_gemini_analysis.txt

[2/3] Detecting floor plan boundaries...

image 1/1 /mnt/c/Users/ryan/Documents/GitHub/construction-ai/datascience/runs/predict/pdf_analysis_complete/example_page1_original.png: 416x640 3 floor_plans, 